# Autonomous Scientific Research Assistant (LangGraph)

In [ ]:
!pip install biopython

### Step 1: Defining the Shared State

In [6]:
from typing import TypedDict, Optional
from langgraph.graph import StateGraph, END

# --- Shared State Schema ---
class ResearchState(TypedDict):
    """
    Maintains the state payload for the Autonomous Scientific Research pipeline.
    Specifically engineered for phytochemical analysis and antibacterial 
    efficacy studies (e.g., Pometia pinnata vs. Aeromonas hydrophila).
    """
    
    # Core Research Query
    research_target: str 
    
    # 1. Raw Data Ingestion (Populated by the Researcher Agent)
    raw_literature_data: str 
    
    # 2. Processed Analytical Data (Populated by the Analyst Agent)
    # Separating the analysis into specific scientific domains based on the KTI methodology
    phytochemical_profile: str      # Tracks active compounds (Flavonoids, Saponins, Tannins)
    extraction_methodology: str     # Tracks solvent methods (e.g., Ethanol maceration)
    inhibition_zone_data: str       # Tracks quantitative antibacterial results (Daya hambat)
    
    # 3. Final Evaluation (Populated by the Critic Agent)
    comparative_analysis: str       # The final synthesis supporting the POMELIQUID hypothesis

### Step 2: Building the PubMed Custom Tool (Researcher)



In [7]:
import logging
from typing import Dict, Any
from Bio import Entrez

# Configure NCBI Entrez connection.
# Professional best practice: Provide email and a custom tool name to prevent rate-limiting.
Entrez.email = "kaylanuansa15@gmail.com" 
Entrez.tool = "PomeliquidResearchAgent" 

def fetch_pubmed_abstracts(query: str, max_results: int = 5) -> str:
    """
    Interfaces with the NCBI PubMed API via E-utilities.
    Retrieves raw abstract payloads for downstream phytochemical and antibacterial analysis.
    """
    logging.info(f"Executing PubMed ESearch for target: [{query}]")
    
    try:
        # ESearch: Fetch specific PubMed IDs (PMIDs) matching the query
        search_handle = Entrez.esearch(db="pubmed", term=query, retmax=max_results)
        search_results = Entrez.read(search_handle)
        search_handle.close()
        
        id_list = search_results.get("IdList", [])
        if not id_list:
            logging.warning(f"No PMIDs resolved for query: {query}")
            return "Insufficient literature found. Consider broadening the botanical or bacterial search terms."
        
        logging.info(f"Resolved {len(id_list)} PMIDs. Initiating payload extraction...")
        
        # EFetch: Retrieve the full abstract text for the identified PMIDs
        fetch_handle = Entrez.efetch(db="pubmed", id=id_list, rettype="abstract", retmode="text")
        abstracts_payload = fetch_handle.read()
        fetch_handle.close()
        
        return abstracts_payload
        
    except Exception as e:
        logging.error(f"NCBI API Connection failed: {e}")
        return f"System Error - Literature retrieval aborted: {e}"

def researcher_node(state: dict) -> Dict[str, Any]:
    """
    Node 1 (Data Ingestion): Acts as the primary research engine.
    Bypasses generalized web search to pull verified, peer-reviewed data 
    specifically related to plant-based antibacterial efficacy.
    """
    logging.info("[NODE EXECUTION] Researcher Agent initializing...")
    
    # Extract the core target from the ResearchState
    target = state.get("research_target", "")
    if not target:
        logging.error("State violation: 'research_target' is null.")
        return {"raw_literature_data": "Error: Null research target."}
    
    # --- The "Plus One" KTI Integration ---
    # We dynamically inject POMELIQUID context to ensure the NCBI database 
    # returns papers relevant to your specific methodology.
    enhanced_query = f"({target}) AND (antibacterial OR Aeromonas hydrophila OR Pometia pinnata OR plant extract)"
    
    # Execute the targeted API call with the enhanced query
    raw_data = fetch_pubmed_abstracts(enhanced_query)
    
    logging.info("[NODE EXECUTION] Researcher Agent completed data ingestion successfully.")
    
    # Map the fetched data directly to the new ResearchState variable
    return {"raw_literature_data": raw_data}

### Step 3: Prompting the Analyst and Critic Agents



In [8]:
import logging
from typing import Dict, Any
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

# Enforce strict zero-temperature for analytical objectivity. 
# This prevents the local LLM from hallucinating fake biological data.
llm = ChatOllama(model="llama3", temperature=0.0)

def analyst_node(state: dict) -> Dict[str, Any]:
    """
    Node 2 (Data Extraction): Parses raw NCBI abstracts through three isolated 
    micro-chains to extract the specific biological metrics required for the 
    POMELIQUID methodology, preventing data cross-contamination.
    """
    logging.info("[NODE EXECUTION] Analyst Agent isolating phytochemical and antibacterial metrics...")
    
    raw_data = state.get("raw_literature_data", "")
    if not raw_data or "Error" in raw_data:
        logging.warning("Insufficient raw data passed to Analyst. Bypassing extraction.")
        return {
            "phytochemical_profile": "No data", 
            "extraction_methodology": "No data", 
            "inhibition_zone_data": "No data"
        }
    
    # Micro-Chain 1: Phytochemical Profiling
    prompt_phyto = ChatPromptTemplate.from_template(
        "You are an analytical Phytochemist. Scan the following NCBI abstracts for mentions of secondary metabolites. "
        "Extract ONLY data regarding flavonoids, saponins, and tannins. If none exist, output 'No data found.'\n\nAbstracts: {data}"
    )
    
    # Micro-Chain 2: Methodological Profiling
    prompt_method = ChatPromptTemplate.from_template(
        "You are a Laboratory Technician. Scan the following NCBI abstracts. "
        "Extract ONLY the extraction methodologies used (e.g., ethanol maceration, solvent concentrations). "
        "If none exist, output 'No data found.'\n\nAbstracts: {data}"
    )
    
    # Micro-Chain 3: Quantitative Antibacterial Profiling
    prompt_zone = ChatPromptTemplate.from_template(
        "You are a Microbiologist. Scan the following NCBI abstracts. "
        "Extract ONLY quantitative data regarding antibacterial inhibition zones (daya hambat) or efficacy against Aeromonas hydrophila. "
        "If none exist, output 'No data found.'\n\nAbstracts: {data}"
    )
    
    # Execute the targeted chains
    logging.info("Executing focused extraction: Phytochemicals...")
    phyto_res = (prompt_phyto | llm).invoke({"data": raw_data}).content
    
    logging.info("Executing focused extraction: Extraction Methods...")
    method_res = (prompt_method | llm).invoke({"data": raw_data}).content
    
    logging.info("Executing focused extraction: Inhibition Zones...")
    zone_res = (prompt_zone | llm).invoke({"data": raw_data}).content
    
    logging.info("[NODE EXECUTION] Analyst Agent completed targeted extractions.")
    
    return {
        "phytochemical_profile": phyto_res.strip(),
        "extraction_methodology": method_res.strip(),
        "inhibition_zone_data": zone_res.strip()
    }

def critic_node(state: dict) -> Dict[str, Any]:
    """
    Node 3 (Synthesis & Peer Review): Evaluates the extracted metrics against 
    the target POMELIQUID hypothesis (efficacy of Pometia pinnata ethanol extract 
    on Aeromonas hydrophila).
    """
    logging.info("[NODE EXECUTION] Critic Agent conducting final comparative analysis...")
    
    prompt_critic = ChatPromptTemplate.from_template(
        "You are a Senior Lead Microbiologist peer-reviewing literature data. "
        "Evaluate the following extracted metrics regarding plant-based antibacterial agents:\n\n"
        "1. Phytochemicals: {phyto}\n"
        "2. Extraction Methodology: {method}\n"
        "3. Inhibition Zones: {zone}\n\n"
        "Task: Synthesize this data into a final 'Comparative Analysis'. Discuss how these findings support or challenge "
        "the hypothesis that ethanol extracts containing flavonoids, saponins, and tannins can act as an effective "
        "biocontrol agent against Aeromonas hydrophila in aquaculture. Format your response professionally using Markdown headers."
    )
    
    chain = prompt_critic | llm
    
    # Pass the three perfectly isolated variables into the final evaluation
    response = chain.invoke({
        "phyto": state.get("phytochemical_profile"),
        "method": state.get("extraction_methodology"),
        "zone": state.get("inhibition_zone_data")
    })
    
    logging.info("[NODE EXECUTION] Critic Agent finalized the comparative analysis.")
    
    return {"comparative_analysis": response.content}

### Step 4: Compiling the LangGraph Pipeline


In [9]:
import logging
from langgraph.graph import StateGraph, END

def build_research_pipeline():
    """
    Constructs and compiles the deterministic StateGraph (DAG) for the research assistant.
    Enforces a strict sequential pipeline to ensure raw NCBI data is fully processed 
    into phytochemical and antibacterial metrics before the final peer review is triggered.
    """
    logging.info("Constructing the POMELIQUID research pipeline graph...")
    
    # Initialize the graph utilizing our custom KTI state schema for strict type-checking
    workflow = StateGraph(ResearchState)

    # Register the specialized biomedical processing nodes
    workflow.add_node("Researcher", researcher_node)
    workflow.add_node("Analyst", analyst_node)
    workflow.add_node("Critic", critic_node)

    # Define the Directed Acyclic Graph (DAG) edges
    # Sequential Flow: Raw Data (NCBI) -> Extracted Variables -> Hypothesis Synthesis
    workflow.set_entry_point("Researcher")
    workflow.add_edge("Researcher", "Analyst")
    workflow.add_edge("Analyst", "Critic")
    workflow.add_edge("Critic", END)

    # Compile the graph into a runnable LangChain application
    compiled_app = workflow.compile()
    
    logging.info("[SYSTEM] LangGraph pipeline compiled successfully.")
    
    return compiled_app

# Instantiate the application for downstream invocation
app = build_research_pipeline()

### Step 5: Running the Autonomous Scientific Research Team

In [10]:
import logging

# Configure professional logging to replace standard print statements
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

def execute_research_pipeline():
    """
    Initializes and executes the Autonomous Scientific Research pipeline.
    This execution specifically targets the antibacterial properties of Pometia pinnata 
    against Aeromonas hydrophila, aligning with the POMELIQUID methodology.
    """
    logging.info("Initializing Autonomous Scientific Research Pipeline...")
    
    # CRITICAL UPDATE: The initial state keys now perfectly match the 
    # ResearchState schema defined in Step 1.
    initial_state = {
        "research_target": "Antibacterial efficacy of Pometia pinnata ethanol extract against Aeromonas hydrophila", 
        "raw_literature_data": "",
        "phytochemical_profile": "",
        "extraction_methodology": "",
        "inhibition_zone_data": "",
        "comparative_analysis": ""
    }
    
    try:
        # Utilizing the new 'research_target' key
        logging.info(f"Target Research Topic: {initial_state['research_target']}")
        logging.info("Invoking LangGraph multi-agent system. This may take a moment...")
        
        # Execute the LangGraph application
        result = app.invoke(initial_state)
        
        # Render the final output with clean, professional terminal formatting
        print("\n" + "="*80)
        print("FINAL SCIENTIFIC REPORT: POMELIQUID HYPOTHESIS".center(80))
        print("="*80 + "\n")
        
        # CRITICAL UPDATE: Fetching 'comparative_analysis' (the output of the Critic node in Step 3)
        # instead of the old 'final_review' key.
        print(result.get("comparative_analysis", "Error: No comparative analysis generated in the state."))
        
        print("\n" + "="*80)
        logging.info("Pipeline execution completed successfully.")
        
    except Exception as e:
        logging.error(f"Pipeline execution failed: {e}")

if __name__ == "__main__":
    execute_research_pipeline()


                 FINAL SCIENTIFIC REPORT: POMELIQUID HYPOTHESIS                 

# Comparative Analysis of Plant-Based Antibacterial Agents

**Introduction**
The study of plant-based antibacterial agents has gained significant attention in recent years, particularly in the context of aquaculture. The hypothesis that ethanol extracts containing flavonoids, saponins, and tannins can act as an effective biocontrol agent against Aeromonas hydrophila is a promising area of research. However, without sufficient data, it is challenging to evaluate the efficacy of these agents.

**Findings**

* **Phytochemicals**: No data was found regarding flavonoids, saponins, and tannins.
* **Extraction Methodology**: No extraction methodologies were extracted from the provided abstracts.
* **Inhibition Zones**: No quantitative data was found regarding antibacterial inhibition zones or efficacy against Aeromonas hydrophila.

**Discussion**
The lack of data on phytochemicals, extraction methodology, and i